# Model Atlas Tool Fault Baseline

## tl;dr

Exact Tool and arguments: 43/72; normal fixture success: 59/72. This is a new narrow baseline, not a historical improvement claim.

## Context & Methods

One actual model output per entry/case/trial is replayed in three modes. Private labels are excluded from the captured requests.

### Key Assumptions

Local, unreviewed exact-literal requests; deterministic handlers; executor retries; two zero-temperature trials are not independent estimates of broad accuracy. No live inference or database writes in this notebook.

## Data

Verify artifact identity, full trial coverage, source hashes and the observation journal.

In [1]:
SOURCE_PATH = 'artifacts/reference-workload/tool-fault-model-baseline-v1.json'
SOURCE_SHA256 = '27975681d2f2b53bad53c44d19346e5d9d7a47fa28f32a7d226faeecd388497f'
AUDIT_PATH = 'docs/reports/2026-09-09_tool_fault_baseline/audit.json'
import hashlib, json, sqlite3
from pathlib import Path
root = Path.cwd()
while not (root / "reference_workload/runtime_matrix.json").exists():
    assert root != root.parent, "repository root not found"
    root = root.parent
source = root / SOURCE_PATH
assert hashlib.sha256(source.read_bytes()).hexdigest() == SOURCE_SHA256
report = json.loads(source.read_text(encoding="utf-8"))
assert report["status"] == "complete"
assert report["human_reviewed"] is False and report["gate_evidence"] is False
assert report["database_writes_performed"] is False
rows = [{"entry": entry["name"], **row} for entry in report["entries"] for row in entry["rows"]]
assert len(rows) == report["protocol"]["expected_observations"]
assert len({(r["entry"], r["id"], r["trial"]) for r in rows}) == len(rows)
assert not any("error" in row for row in rows)
for name, expected in report["source_sha256"].items():
    assert hashlib.sha256((root / "backend/app" / name).read_bytes()).hexdigest() == expected, name
journal = source.with_suffix(".observations.jsonl")
assert hashlib.sha256(journal.read_bytes()).hexdigest() == report["journal_sha256"]
assert [json.loads(line) for line in journal.read_text(encoding="utf-8").splitlines()] == rows
for filename, key in [("runtime_matrix.json", "matrix_sha256"),
                      ("diagnostics/tool-fault-baseline-v1.json", "case_pack_sha256"),
                      ("revisions/1.0.4/manifest.json", "manifest_sha256")]:
    assert hashlib.sha256((root / "reference_workload" / filename).read_bytes()).hexdigest() == report[key]
print({"observations": len(rows), "journal_and_source_hashes": "verified"})

{'observations': 72, 'journal_and_source_hashes': 'verified'}


### Independently Check Requests And Traces

Recompute exact calls from captured JSON and private labels, and verify pairing and handler counts.

In [2]:
def digest(value):
    return hashlib.sha256(json.dumps(value, ensure_ascii=False, sort_keys=True,
                                    separators=(",", ":")).encode()).hexdigest()
cases = {c["id"]: c for c in report["case_pack"]["cases"]}
flat = []
faults = []
for entry in report["entries"]:
    expected_keys = {(case_id, trial) for case_id in cases
                     for trial in range(1, entry["matrix_entry"]["trials"] + 1)}
    assert {(r["id"], r["trial"]) for r in entry["rows"]} == expected_keys
    assert entry["generation_config"] == entry["matrix_entry"]["generation"]
for row in rows:
    body = row["request_body"]
    assert row["request_hash"] == digest(body)
    assert body["seed"] == row["seed"] == 41 + row["trial"]
    public = json.loads(body["messages"][1]["content"])
    assert public["input"]["query"] == cases[row["id"]]["request"]
    assert "expected_tool_schema_json" not in public
    assert "tool_fault_scenario" not in public["input"]
    assert "simulate_failure" not in json.dumps(public["input"]["available_tools"])
    normalized = json.loads(row["normalized_output"])
    tool, arguments = normalized["tool_name"], normalized["arguments"]
    selected = tool == cases[row["id"]]["expected_tool"]
    exact = arguments == cases[row["id"]]["expected_arguments"]
    assert selected == row["selection_correct"]
    assert exact == row["arguments_exact"]
    assert (selected and exact) == row["selection_and_arguments_exact"]
    guard = row["metadata"]["tool_call_contract"]
    assert digest(arguments) == guard["normalized_arguments_hash"]
    assert guard["raw_arguments_hash"] == guard["normalized_arguments_hash"]
    assert row["raw_schema_valid"] == row["raw_guard"]["schema_valid"]
    assert row["normalized_schema_valid"] == guard["schema_valid"]
    assert row["guard_eligible"] == (not guard["schema_errors"] and not guard["boundary_errors"])
    for mode, trace in row["traces"].items():
        assert trace["steps"][0]["arguments"] == arguments
        attempts = trace["steps"][0]["attempts"]
        scenario = trace["fault_scenario"]
        assert scenario["gate_evidence"] is False
        assert scenario["handler_invocation_count"] == sum(a["handler_invoked"] for a in attempts)
        assert scenario["injected_failure_count"] == sum(a["fault_injected"] for a in attempts)
        if not guard["execution_allowed"] or not selected:
            assert attempts == [] and scenario["status"] == "not_exercised"
        if mode == "permanent":
            assert trace["successful"] is False
        faults.append((row["entry"], mode, int(trace["successful"]), int(scenario["passed"]),
                       int(scenario["status"] == "not_exercised"),
                       scenario["handler_invocation_count"], scenario["injected_failure_count"]))
    flat.append((row["entry"], row["id"], row["trial"], int(selected),
                 int(row["raw_schema_valid"]), int(row["guard_eligible"]), int(exact),
                 int(selected and exact), int(row["traces"]["normal"]["successful"])))
connection = sqlite3.connect(":memory:")
connection.row_factory = sqlite3.Row
connection.execute("CREATE TABLE observations (entry, case_id, trial, selected, schema_valid, eligible, exact_arguments, exact_call, tool_success)")
connection.executemany("INSERT INTO observations VALUES (?,?,?,?,?,?,?,?,?)", flat)
connection.execute("CREATE TABLE faults (entry, mode, success, passed, not_exercised, handlers, injected)")
connection.executemany("INSERT INTO faults VALUES (?,?,?,?,?,?,?)", faults)
print({"paired_traces": len(faults), "independent_row_checks": "passed"})

{'paired_traces': 216, 'independent_row_checks': 'passed'}


## Results

SQLite aggregates the independently reconstructed rows, then reconciles the runner summaries.

In [3]:
SQL = 'SELECT entry, COUNT(*) AS total, SUM(selected) AS selected,\nSUM(schema_valid) AS schema_valid, SUM(eligible) AS eligible,\nSUM(exact_call) AS exact_call, SUM(tool_success) AS tool_success,\nSUM(tool_success = 1 AND exact_call = 0) AS success_with_wrong_values\nFROM observations GROUP BY entry ORDER BY entry'
summary = [dict(r) for r in connection.execute(SQL)]
for item in summary:
    original = next(e["summary"] for e in report["entries"] if e["name"] == item["entry"])
    for independent, stored in [("total", "observation_count"), ("selected", "selection_correct"),
                               ("schema_valid", "raw_schema_valid"), ("eligible", "guard_eligible"),
                               ("exact_call", "selection_and_arguments_exact"),
                               ("tool_success", "normal_tool_success")]:
        assert item[independent] == original[stored]
print(json.dumps(summary, ensure_ascii=False, indent=2))
mode_summary = [dict(r) for r in connection.execute(
    "SELECT mode, COUNT(*) AS total, SUM(success) AS successes, SUM(passed) AS fixture_passed, "
    "SUM(not_exercised) AS not_exercised, SUM(handlers) AS handlers, SUM(injected) AS injected "
    "FROM faults GROUP BY mode ORDER BY mode")]
print(json.dumps(mode_summary, indent=2))
output = {"source_sha256": SOURCE_SHA256, "summary": summary, "mode_summary": mode_summary,
          "checks": "passed", "observation_count": len(rows), "paired_trace_count": len(faults)}
(root / AUDIT_PATH).write_text(json.dumps(output, ensure_ascii=False, indent=2), encoding="utf-8")

[
  {
    "entry": "medium-candidate",
    "total": 24,
    "selected": 24,
    "schema_valid": 24,
    "eligible": 24,
    "exact_call": 16,
    "tool_success": 24,
    "success_with_wrong_values": 8
  },
  {
    "entry": "prompt-variant",
    "total": 24,
    "selected": 20,
    "schema_valid": 24,
    "eligible": 24,
    "exact_call": 14,
    "tool_success": 20,
    "success_with_wrong_values": 6
  },
  {
    "entry": "small-baseline",
    "total": 24,
    "selected": 15,
    "schema_valid": 24,
    "eligible": 24,
    "exact_call": 13,
    "tool_success": 15,
    "success_with_wrong_values": 2
  }
]
[
  {
    "mode": "normal",
    "total": 72,
    "successes": 59,
    "fixture_passed": 59,
    "not_exercised": 13,
    "handlers": 59,
    "injected": 0
  },
  {
    "mode": "permanent",
    "total": 72,
    "successes": 0,
    "fixture_passed": 59,
    "not_exercised": 13,
    "handlers": 0,
    "injected": 59
  },
  {
    "mode": "transient_once",
    "total": 72,
    "successes": 5

1417

## Takeaways

Use this as a frozen diagnostic baseline. Execution success does not prove requested-value correctness. Permanent expected failure is never Tool success. Review explicit value failures before changing the generation strategy; retain this baseline and do not promote it to Gate evidence.